# AWS Shared Responsibility Model: who secures what

**Goal:** by the end of this lesson you can explain which security jobs belong to AWS and which belong to you, see how this changes from one service to another, check your own side of an AWS account with the CLI, and see one real example with an S3 bucket.

| Part | Topic |
|---|---|
| 0 | Set up the CLI inside the notebook |
| 1 | The big picture (an apartment building) |
| 2 | Security **of** the cloud and security **in** the cloud |
| 3 | A small game: who is responsible? (no AWS needed) |
| 4 | It changes with the service (EC2, RDS, S3, Lambda) |
| 5 | Shared controls, and how to see AWS's proof (AWS Artifact) |
| 6 | Look at **your** side of your account (read-only) |
| 7 | Hands-on: an S3 bucket, AWS defaults and your choices |
| 8 | Clean-up |
| 9 | Exercises |

**Cost:** almost zero. Parts 0 to 6 only read information. Part 7 creates one empty S3 bucket with one tiny file, and Part 8 deletes both.

### How to use this notebook
- A notebook has **text cells** (like this one) and **code cells**.
- To run a code cell: click it and press **Shift + Enter**.
- Run the cells **in order**, from top to bottom.
- When a line starts with **`!`**, it is a terminal command: exactly what you would type in your computer's terminal, just without the `!`.

---
## Part 0: Set up the CLI

The **AWS CLI** is a program that lets you give orders to AWS by typing commands instead of clicking around the website (the *console*).

The cell below checks whether the CLI is already installed. If it is not, it installs it (this automatic install only works on Linux, e.g. Google Colab. On your own computer, follow the README).

In [ ]:
!aws --version 2>/dev/null || (curl -s "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o /tmp/awscliv2.zip && unzip -q -o /tmp/awscliv2.zip -d /tmp && /tmp/aws/install --update && aws --version)

### Log in (without leaving your keys in the notebook)

The CLI needs your *access key ID* and your *secret access key* to know **who you are**.

⚠️ **Never paste your keys straight into a cell.** The notebook is saved and may be shared. That is why we use `getpass`, which asks for the key in a hidden box and saves nothing in the file.

In [ ]:
import os, json, subprocess
from getpass import getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass("Access key ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Secret access key: ")
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"   # Ireland. The CLI always needs a "home" Region.
os.environ["AWS_PAGER"] = ""                      # prints everything at once, without paging

print("Done! Keys kept only in this session's memory.")

First test: ask AWS **"who am I?"**. If you see your account number and your user, everything works.

In [ ]:
!aws sts get-caller-identity

Two small helpers. The first runs a command and gives the answer back as Python data. The second runs a command and gives back the text, even when AWS says *no* (we will need that later). You do not need to understand their code right now:

In [ ]:
def aws_json(command):
    '''Runs a CLI command and returns the answer as Python data.'''
    r = subprocess.run("aws " + command + " --output json", shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
    return json.loads(r.stdout) if r.stdout.strip() else {}

def aws_try(command):
    '''Runs a CLI command and returns (worked?, text). It never stops the notebook.'''
    r = subprocess.run("aws " + command, shell=True, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()

ACCOUNT_ID = aws_json("sts get-caller-identity")["Account"]
print("Helpers ready. Your account number ends in ...", ACCOUNT_ID[-4:])

---
## Part 1: The big picture

Think of an **apartment building**.

| In the building | In AWS |
|---|---|
| The **building owner** | **AWS** |
| The walls, the roof, the elevator, the guard at the front door, the power and water | The **data centers**, servers, cables, power, cooling and the guards |
| **Your apartment** | **Your AWS account** and everything you put in it |
| The **lock on your door** and who gets a key | **Your** permissions, passwords and firewall rules |
| The **things inside** your apartment | **Your data** and your applications |

The owner keeps the **building** safe. But if you leave your own door open, you cannot blame the owner.

This split is called the **Shared Responsibility Model**: security is a job shared between AWS and you, and each side has its own part.

---
## Part 2: Security *of* the cloud and security *in* the cloud

AWS uses two short phrases:

| | Who | What it means | Examples |
|---|---|---|---|
| **Security OF the cloud** | **AWS** | Protecting the base that every service runs on | Data center buildings and guards, power, cooling, servers, network cables, the software that runs virtual machines (the *hypervisor*), Regions, AZs and edge locations, destroying old disks |
| **Security IN the cloud** | **You** | Protecting what you put in the cloud and how you set it up | Your data, who can access it (IAM), passwords and MFA, firewall rules (Security Groups), updates of **your** operating system and apps, encryption choices, backups |

A simple way to remember it:
- **AWS** protects the **cloud itself** (the building).
- **You** protect **what you do with it** (your apartment).

### Two important details
1. **AWS gives you tools, but you decide to use them.** AWS offers encryption, MFA, firewalls and logs. Turning them on, and using them well, is **your** job.
2. **Your data is always yours.** No service moves the responsibility for your data and for who can read it to AWS.

---
## Part 3: A small game: who is responsible?

No AWS account needed here. Below are 10 real tasks. For each one, decide who does it:
- `"AWS"` if it is security **of** the cloud,
- `"YOU"` if it is security **in** the cloud.

Run the first cell to see the tasks.

In [ ]:
TASKS = [
    ("Guards at the door of the data center",                                  "AWS", "Physical security of the buildings is always AWS."),
    ("Install security updates on the operating system of YOUR EC2 server",    "YOU", "On EC2 you manage the guest operating system."),
    ("Decide who can read the files in your S3 bucket",                        "YOU", "Access to your data is always your decision."),
    ("Keep power and cooling running in the data center",                      "AWS", "Part of the facilities."),
    ("Turn on MFA for your IAM users",                                         "YOU", "AWS offers MFA; turning it on is your job."),
    ("Update the software that runs the virtual machines (the hypervisor)",    "AWS", "It sits below your servers. You cannot even see it."),
    ("Open or close ports in a Security Group",                                "YOU", "It is your firewall."),
    ("Choose to encrypt your data",                                            "YOU", "AWS gives the tools; you choose to use them."),
    ("Destroy old storage devices so the data cannot be recovered",            "AWS", "AWS handles the life of its hardware."),
    ("Create a backup plan for your database",                                 "YOU", "AWS can run backups, but you decide how often and how long to keep them."),
]

for n, (task, _, _) in enumerate(TASKS, start=1):
    print(f"{n:>2}. {task}")

Now it is your turn. Change each `"?"` to `"AWS"` or `"YOU"`, then run the cell:

In [ ]:
my_answers = {1: "?", 2: "?", 3: "?", 4: "?", 5: "?", 6: "?", 7: "?", 8: "?", 9: "?", 10: "?"}

score = 0
for n, (task, right, why) in enumerate(TASKS, start=1):
    guess = my_answers[n].strip().upper()
    if guess == right:
        score += 1
        print(f"{n:>2}. OK    {right}")
    else:
        print(f"{n:>2}. WRONG (you said {guess}) -> {right}. {why}")
print(f"\nScore: {score}/{len(TASKS)}")

---
## Part 4: It changes with the service

The line between AWS and you **moves**. The more AWS runs for you, the shorter your list gets.

| Service | What AWS also does | What stays with you |
|---|---|---|
| **Amazon EC2** (a virtual server) | Hardware, hypervisor | The operating system and its updates, firewall rules, your apps, your data, access |
| **Amazon RDS** (a managed database) | Also the operating system and the database software updates (you choose *when*, in a maintenance window) | Who can connect, database users, your data, backup settings, encryption choice |
| **Amazon S3** (file storage) | The whole storage system | Who can read and write, public access settings, encryption choice, versioning |
| **AWS Lambda** (runs your code, no server to manage) | Servers, operating system, runtime updates | Your code, the permissions you give it, the data it handles |

Look at the pattern: on the **left side** AWS does more and more. But the **right side never becomes empty**. **Data and access always stay with you.**

The cell below turns the right column into a simple picture. The lists are the ones from the table above. It is a teaching picture, **not** an official count:

In [ ]:
your_side = {
    "Amazon EC2": ["Operating system updates", "Firewall rules (Security Groups)", "Your applications", "Your data", "Who can access it (IAM)", "Backups"],
    "Amazon RDS": ["Who can connect", "Database users and passwords", "Your data", "Backup settings", "Encryption choice"],
    "Amazon S3":  ["Who can read and write", "Public access settings", "Encryption choice", "Versioning"],
    "AWS Lambda": ["Your code", "Permissions of the function", "The data it handles"],
}

for service, things in your_side.items():
    print(f"{service:<11} {'█' * len(things):<7} {len(things)} things on YOUR list")
    for t in things:
        print("      -", t)

---
## Part 5: Shared controls, and how to see AWS's proof

### Shared controls
Some jobs have a part for AWS **and** a part for you. AWS calls them **shared controls**:

| Control | AWS part | Your part |
|---|---|---|
| **Patch management** (updates) | Fixes problems in the cloud infrastructure | Fixes your guest operating system and your applications |
| **Configuration management** | Configures its own devices and services | Configures your servers, databases and applications |
| **Awareness and training** | Trains AWS employees | Trains your own team |

### "But is AWS really doing its part?"
You can check. **AWS Artifact** is a free page in the AWS console where you can download AWS's audit reports and certificates (for example SOC reports and ISO certificates). Auditors and security teams use it as proof of AWS's side.

⚠️ **Important:** AWS being certified does **not** make *your* application certified. Your side still needs to be set up correctly.

---
## Part 6: Look at **your** side of your account (read-only)

Now we use the CLI to look at things that belong to **you**. These commands **only read**. They change nothing.

(If a command says "access denied", your user does not have permission to read that item. That is fine, just go on.)

### 6.1 Does the root user have MFA? Does it have access keys?
The *root user* is the all-powerful login created with the account. It should have **MFA on** and **no access keys**.

In [ ]:
summary = aws_json("iam get-account-summary")["SummaryMap"]
print("Root user has MFA on     :", bool(summary["AccountMFAEnabled"]), "  (should be True)")
print("Root user has access keys:", bool(summary["AccountAccessKeysPresent"]), "  (should be False)")

### 6.2 Which IAM users do **not** have MFA?
Users with a password and no MFA are an easy target.

In [ ]:
users = aws_json("iam list-users")["Users"]
no_mfa = []
for u in users:
    devices = aws_json(f"iam list-mfa-devices --user-name {u['UserName']}")["MFADevices"]
    if not devices:
        no_mfa.append(u["UserName"])

print(f"IAM users in this account: {len(users)}")
print("Users WITHOUT an MFA device:", no_mfa if no_mfa else "none")

### 6.3 Is there a password policy?
A *password policy* sets the rules for IAM passwords (length, symbols, how often to change). AWS does not force one on you: it is **your** choice.

In [ ]:
ok, out = aws_try("iam get-account-password-policy")
if ok:
    print(out)
elif "NoSuchEntity" in out:
    print("No custom password policy yet. AWS did not set one for you: this is on YOUR side.")
else:
    print(out)

### 6.4 Which firewall rules are open to the whole internet?
A *Security Group* is the firewall of your servers. The address `0.0.0.0/0` means **everyone on the internet**. That is fine for a public website (ports 80 and 443), but **not** for things like SSH (port 22) or databases.

In [ ]:
groups = aws_json("ec2 describe-security-groups --filters Name=ip-permission.cidr,Values=0.0.0.0/0")["SecurityGroups"]

found = 0
for g in groups:
    for rule in g["IpPermissions"]:
        if any(r.get("CidrIp") == "0.0.0.0/0" for r in rule.get("IpRanges", [])):
            ports = "all ports" if rule.get("IpProtocol") == "-1" else f"{rule.get('FromPort')}-{rule.get('ToPort')}"
            print(f"{g['GroupId']:<22} {g['GroupName']:<25} open to the world on {ports}")
            found += 1

print(f"\nRules open to 0.0.0.0/0 in eu-west-1: {found}")

### 6.5 Is S3 "Block Public Access" on for the whole account?
This is a safety switch. When it is on, no bucket in the account can be made public by mistake. Turning it on is **your** choice.

In [ ]:
ok, out = aws_try(f"s3control get-public-access-block --account-id {ACCOUNT_ID}")
if ok:
    print(out)
elif "NoSuchPublicAccessBlockConfiguration" in out:
    print("No account-level block configured. New buckets are still protected by default (see Part 7),")
    print("but turning it on for the whole account is your decision.")
else:
    print(out)

**What did you learn?** Every item in this part is on **your** side of the line. AWS gave you the tools (MFA, password policy, Security Groups, Block Public Access). Using them is up to you.

---
## Part 7: Hands-on: an S3 bucket, AWS defaults and your choices

We will see both sides in one small example. We create a bucket, look at what **AWS already did for us**, and then try to do something unsafe to see what **happens**.

### 7.1 Create a bucket and a small file
The bucket name gets a random ending, because bucket names must be unique in the whole world.

In [ ]:
import uuid
REGION = "eu-west-1"
BUCKET = "study-shared-resp-" + uuid.uuid4().hex[:10]
print("Bucket name:", BUCKET)

!aws s3api create-bucket --bucket {BUCKET} --region {REGION} --create-bucket-configuration LocationConstraint={REGION}

with open("/tmp/hello.txt", "w") as f:
    f.write("hello from the shared responsibility lesson\n")
!aws s3 cp /tmp/hello.txt s3://{BUCKET}/hello.txt

### 7.2 What did **AWS** do for us?
Two defaults that AWS turns on for every new bucket:
- **Block Public Access**: all four switches should say `true`.
- **Encryption**: files are encrypted at rest with keys that AWS manages (SSE-S3).

We did not ask for either one.

In [ ]:
!aws s3api get-public-access-block --bucket {BUCKET}
!aws s3api get-bucket-encryption --bucket {BUCKET} --query "ServerSideEncryptionConfiguration.Rules[].ApplyServerSideEncryptionByDefault"

### 7.3 What happens if **we** try to make the bucket public?
Let us write a policy that lets **everyone in the world** read the files, and try to attach it. (Nothing will become public: watch the answer.)

In [ ]:
public_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "PublicRead",
        "Effect": "Allow",
        "Principal": "*",
        "Action": "s3:GetObject",
        "Resource": f"arn:aws:s3:::{BUCKET}/*",
    }],
}
with open("/tmp/public-policy.json", "w") as f:
    json.dump(public_policy, f)

ok, out = aws_try(f"s3api put-bucket-policy --bucket {BUCKET} --policy file:///tmp/public-policy.json")
print("Worked?", ok)
print(out)

You should see **AccessDenied** and a mention of **BlockPublicPolicy**. AWS refused, because the safety switch is on.

Now think about what this shows:
- **AWS's side:** it built a safety net and turned it on by default.
- **Your side:** you hold the switches. You *can* turn Block Public Access off, and then the same policy would work. If you do that by mistake and the data leaks, the cause is **your configuration**, not a failure of AWS.

⚠️ We will **not** turn it off. Do not try this with real data.

---
## Part 8: Clean-up

Delete everything this lesson created, in the right order: first the file, then the bucket, then the small files on this computer. After that we remove the keys from this notebook's memory.

In [ ]:
!aws s3 rm s3://{BUCKET} --recursive
!aws s3api delete-bucket --bucket {BUCKET} --region {REGION}

for path in ["/tmp/hello.txt", "/tmp/public-policy.json"]:
    if os.path.exists(path):
        os.remove(path)

ok, out = aws_try(f"s3api head-bucket --bucket {BUCKET}")
print("Bucket still exists?", ok, "  (should be False)")

In [ ]:
for name in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]:
    os.environ.pop(name, None)
print("Keys removed from this session's memory.")

---
## Part 9: Exercises

1. For each task, say who is responsible, **AWS** or **you**:
   - a) Replace a broken hard disk in a data center.
   - b) Install a security update on the operating system of your EC2 server.
   - c) Install a security update on the operating system of your RDS database.
   - d) Decide which IP addresses can reach your database.
2. Your company uses **AWS Lambda** instead of EC2. Name two things that **leave** your list, and two that **stay**.
3. A news story says: *"Company X lost customer data because AWS was hacked."* The real cause was an S3 bucket left public. Who was responsible for that setting? Why?
4. Look at the result of Part 6.4 for your account. Is there any rule open to `0.0.0.0/0` on port 22 (SSH) or on a database port? What would be a safer rule?
5. **Challenge:** a friend says *"AWS has the SOC 2 certificate, so my app is certified too."* Explain, in two sentences, why this is wrong. Where can you download AWS's reports?

<details>
<summary>Answers</summary>

1. a) **AWS** (hardware). b) **You** (guest OS on EC2). c) **AWS** (RDS is managed; you only choose the maintenance window). d) **You** (network access rules).
2. **Leave:** the operating system and its updates, managing servers. **Stay:** your code, the permissions of the function (IAM role), and your data.
3. **The company.** Who can access a bucket is a setting on the customer's side. AWS provides Block Public Access, but the customer holds the switch.
4. Port 22 or a database port open to `0.0.0.0/0` is risky. Safer: allow only your own IP address or your company network, or use a bastion host or AWS Systems Manager Session Manager instead of open SSH.
5. AWS's certificate covers **AWS's part** (the cloud itself). Your application, your settings and your data are your part and need their own controls. AWS's reports are in **AWS Artifact** (free, in the console).

</details>

---
## Cheat sheet

| Term | In one sentence |
|---|---|
| **Shared Responsibility Model** | Security is shared: AWS protects the cloud, you protect what you put in it |
| **Security OF the cloud** | AWS: buildings, hardware, network, hypervisor, Regions, AZs, edge locations |
| **Security IN the cloud** | You: your data, access (IAM, MFA), firewall rules, guest OS updates, encryption choices |
| **Changes with the service** | More managed service (RDS, S3, Lambda) = less on your list. **Data and access always stay yours** |
| **EC2** | You manage the operating system and its updates |
| **RDS** | AWS patches the OS and database software; you choose the time and manage access and data |
| **Lambda** | You bring the code and the permissions; AWS runs everything else |
| **Shared controls** | Patching, configuration and training: AWS does its part, you do yours |
| **AWS Artifact** | Free place in the console to download AWS's audit reports and certificates |
| **Block Public Access** | S3 safety switch, on by default for new buckets. You can turn it off, so be careful |
| **Least privilege** | Give each person only the permissions they need (your side, with IAM) |